## Feature Engineering

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/tracking_ice_processed.csv')

### Exploratory Data Analysis
Since we want to predict the *next* zip code, we need to make the following transformations:
- `month` column: transform from the strings of months to datetime objects
- `year` column: transform from the integer year to datetime objects

By making the above transformations, we introduce the concept of time.

The challenge is that our Target Variable `zip_code` is a Categorical/Nominal Feature, in numerical format.

In [ ]:
# Checking the overall DataFrame
df.info()

# Printing the first 5 rows to get an idea of what the data looks like
display(df.head(5))

print(df['city'].nunique())
print(df['zip_code'].nunique())


<class 'pandas.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   month           169 non-null    str  
 1   event           169 non-null    str  
 2   day_with_month  169 non-null    str  
 3   city            169 non-null    str  
 4   zip_code        169 non-null    int64
 5   year            169 non-null    int64
dtypes: int64(2), str(4)
memory usage: 8.1 KB


,month,event,day_with_month,city,zip_code,year
0,January,"On January 7, the day that the United States C...",January 7,"Kern County, California",93387,2025
1,January,"On January 23, high-profile US Immigration and...",January 23,"Atlanta, Georgia",30301,2025
2,January,"On January 23, high-profile US Immigration and...",January 23,"Boston, Massachusetts",2101,2025
3,January,"On January 23, high-profile US Immigration and...",January 23,"Denver, Colorado",80201,2025
4,January,"On January 23, high-profile US Immigration and...",January 23,"Miami, Florida",33101,2025


<StringArray>
[                'Kern County, California',
                        'Atlanta, Georgia',
                   'Boston, Massachusetts',
                        'Denver, Colorado',
                          'Miami, Florida',
                 'New York City, New York',
                      'Newark, New Jersey',
              'Philadelphia, Pennsylvania',
                     'Seattle, Washington',
                        'Washington, D.C.',
 ...
                   'Oakmont, Pennsylvania',
        'Lewiston, Maine\nPortland, Maine',
                         'Lewiston, Maine',
                'Cumberland County, Maine',
                      'York County, Maine',
              'Phoenixville, Pennsylvania',
 'Lower Providence Township, Pennsylvania',
               'Santa Barbara, California',
                       'Buffalo, New York',
                     'Hayward, California']
Length: 107, dtype: str
104


In [ ]:
# Combining the year and month day into a single column, to be able to cast as a datetime object
df['month_day_year'] = df['day_with_month'].str.cat(df['year'].astype(str), sep=' ')

# Dropping the month and year columns now, since we have a combined column to capture the data
df.drop(columns=['month', 'day_with_month', 'year'], inplace=True)

In [ ]:
# Converting the month_day_year column from string to datetime
df["date"] = pd.to_datetime(df["month_day_year"], format="%B %d %Y")

# Dropping the month_day_year now after creating the new column
df.drop(columns=["month_day_year"], inplace=True)

In [ ]:
# Ensure that the DataFrame is sorted by the date column
df.sort_values(by='date', inplace=True)

In [ ]:
# In order to create a chronological order, we want to keep track of the zip_code of the previous event
df['previous_zip_code'] = df['zip_code'].shift(1)   # There's already a built-in function that does this!!!!!!
df['previous_zip_code'] = df['previous_zip_code'].astype('Int64')

In [ ]:
# Finally, dropping the city column as we don't want the model to have the answer already
df.drop(columns=['city'], inplace=True)

In [ ]:
# Dropping the VERY first row, because it contains the previous_zip_code as a null value
# This null value could introduce errors or bad training if incorporated into the training dataset, so it's safest to simply drop it
df.dropna(inplace=True)

In [ ]:
df.info()

display(df.head(10))

## Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# One Hot Encoding our previous_zip_code column. This is because we want to make sure the model treats these Features as categorical, and not ordinal
dummy_variables = pd.get_dummies(df['previous_zip_code'], prefix="is", drop_first=False).astype(int)

# Now, combining the One Hot Encoded columns back to the original dataframe, but dropping the previous_zip_code column
# This helps by converting our Features into 0's or 1's, which our model understands better
df_encoded = pd.concat([df.drop(columns=['previous_zip_code']), dummy_variables], axis=1)

# Splitting our data into X and y, where X is all of our Features and y is our target variable
# Dropping zip_code because it is the Target Variable
# Dropping event because it is simply a long string description.
X = df_encoded.drop(columns=["zip_code", "event", "date"])
y = df_encoded["zip_code"]

# Creating our test/train data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=23,
    shuffle=False   # Shuffle FALSE becase we want to keep the chronological order
)

# Instantiating our RandomForestClassifier model
rf_model = RandomForestClassifier()

# Training our model (we DON'T Transform on Categorical values, only Fit!)
rf_model.fit(X_train, y_train)

# Making predictions
y_pred = rf_model.predict(X_test)



In [ ]:
# Checking the Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

## Reflection
Okay, so it looks like the current Features selected may not be a good idea. Model Accuracy of 5% is not good. The problem may be that with such low number of observations, Zip Codes may not be the correct Target Variable here.